# Final evaluation: probability distribution and calibration

This notebook does not retrain anything. It only loads the artifact and figures produced by `src/final_model.py`, `src/plot_final_probabilities.py` and `src/plot_calibration.py`, and documents what they show about the final model closed on 2026-07-19.

**Model:** `GradientBoostingClassifier(random_state=42)`. **Features (7):** `Type_L`, `Type_M`, `Temperature difference`, `Rotational speed [rpm]`, `Power [W]`, `Tool wear [min]`, `OSF criterion` (raw `Torque [Nm]` and `Process temperature [K]` removed as redundant). **Threshold:** cost-dependent, working recommendation 0.30 for a typical 5-10x FN:FP ratio. Artifact: `models/final_model.joblib`.

## Probability distribution

Predicted probability of failure on `X_test`, split by true class (see `src/plot_final_probabilities.py`):

![Probability distribution](../results/final_model_probability_distribution.png)

The two classes are strongly separated (most non-failures near 0.0, most failures near 1.0). Very few records fall between the candidate thresholds 0.30 and 0.50, which is why the choice of threshold in that range does not swing the error counts drastically.

## Calibration check

Reliability diagram comparing two probability sources (see `src/plot_calibration.py`):

- **OOF (train)** — out-of-fold probabilities from the same 5-fold CV used to pick the threshold in `final_model.py`. Never touches `X_test`.
- **X_test** — the fitted artifact model scored on the held-out development check set (small, and already used during exploration).

![Calibration](../results/final_model_calibration.png)

Marker area is scaled by the number of records in each bin (`results/final_model_calibration.csv`), because the bins are very unevenly populated:

| source | mean_predicted | fraction_positive | count |
|---|---|---|---|
| OOF (train) | 0.0035 | 0.0040 | 7699 |
| OOF (train) | 0.1386 | 0.0513 | 39 |
| OOF (train) | 0.2418 | 0.0000 | 5 |
| OOF (train) | 0.3562 | 0.4615 | 13 |
| OOF (train) | 0.4563 | 0.3000 | 10 |
| OOF (train) | 0.5353 | 0.0000 | 6 |
| OOF (train) | 0.6589 | 0.6000 | 5 |
| OOF (train) | 0.7415 | 0.6667 | 3 |
| OOF (train) | 0.8573 | 1.0000 | 15 |
| OOF (train) | 0.9666 | 0.9854 | 205 |

**Brier score:** 0.0057 (OOF) / 0.0050 (X_test) — both low, consistent with the strong class separation seen above.

**Reading this diagram:**

- The dense bins (near 0.0 and near 1.0, thousands/hundreds of records) sit almost exactly on the diagonal — the model's confidence is well calibrated where most of the data lives.
- The middle bins (0.2-0.8, where the decision thresholds 0.30/0.50 sit) are populated by only 3-15 records each. The visible zig-zag there is small-sample noise, not evidence of miscalibration — with that few records, a couple of extra failures or non-failures shifts the point a lot.
- **Conclusion:** the model is well calibrated overall (low Brier score, good behaviour at the extremes), but the calibration *in the exact region where the threshold decision is made* cannot be assessed reliably from this dataset — there simply aren't enough borderline records. This is a caveat for the threshold choice, not a reason to recalibrate (e.g. with `CalibratedClassifierCV`) based on this data alone.

## Drift-monitoring tool (sanity check only, no production data yet)

There is no production data stream yet to check for real drift against. What
exists instead is the tool itself (`src/drift_monitoring.py`) plus a
sanity check (`src/check_drift.py`) comparing `X_train` (reference) vs
`X_test` (current) — since both come from the same stratified split, this
should show no meaningful drift. A passing sanity check means the tool is
trustworthy to point at a real new data batch once one exists.

Two metrics per feature: **PSI** (Population Stability Index, numeric
features, binned on reference deciles; < 0.1 no shift, 0.1-0.25 moderate,
> 0.25 significant) and **proportion difference** for the binary
categorical features `Type_L`/`Type_M` (PSI's decile binning does not apply
to a two-valued feature).

![Drift sanity check](../results/drift_check_train_vs_test.png)

All numeric features land well under the 0.1 PSI line, and the categorical
proportion differences are near zero — as expected for two samples of the
same distribution. This confirms the metric behaves correctly; it says
nothing yet about real-world drift.

**To use this for real once production data exists:** load the new batch in
place of `X_test_raw.csv` in `check_drift.py`, keep `X_train_raw.csv` as the
reference, and re-run. A PSI above 0.25 on a feature that matters to the
model (see the feature-importance ranking: Power, rpm, Temperature
difference, OSF criterion) would be the trigger to investigate before
trusting the model's predictions on that batch.

## Open items

- **Real FN:FP business costs** — needed to move from the working recommendation (threshold 0.30) to a final decision.
- **Validation on untouched/future data** — `X_test` was used during exploration and OSF design; treat current metrics as a development check, not an external validation.
- **Drift monitoring against real production data** — the tool above is ready; it just has nothing to monitor until production data starts arriving.